# Down-and-in put

The capital-at-risk put of an autocallable, $\mathbb 1\{S_T < B\}\,(K_0 - S_T)^+/K_0$: the barrier is observed **at expiry
only**. Priced with the `AutocallableOption` template (no coupons, autocall disabled) against `BarrierOption` (barrier at
expiry, then discrete and continuous monitoring). Docs: *Models and Pricing* (Products: Barrier options, Autocallable).

In [ ]:
import os
from pathlib import Path
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

REPO = Path.cwd().resolve().parents[1]

import numpy as np
import pandas as pd
import tensorflow as tf
import tensorquant as tq

## Market

In [ ]:
mkt = tq.MarketEnvironment.from_folder(REPO / "marketdata")   # latest snapshot
EVALUATION_DATE = tq.Settings.evaluation_date = mkt.evaluation_date
TICKER = "TICKER1"
eq = mkt.get_equity(TICKER)
K0 = round(float(eq.spot), 2)                                 # initial fixing
calendar, mf = tq.TARGET(), tq.BusinessDayConvention.ModifiedFollowing
print(f"{EVALUATION_DATE}  {TICKER} spot {float(eq.spot):.2f}")

## Put
5Y, strike $K_0$, barrier 80%, participation 100%; values in % of the notional ($= K_0$ shares).

In [ ]:
BARRIER_PCT = 80.0
H = BARRIER_PCT / 100 * K0
expiry = calendar.advance(EVALUATION_DATE, 5, tq.TimeUnit.Years, mf)


def dip(monitoring_dates=None):
    return tq.BarrierOption(eq.currency, EVALUATION_DATE, expiry, tq.OptionType.Put, K0, H, tq.BarrierType.DownIn,
                            monitoring_dates=monitoring_dates, underlying=TICKER)


def pct(pricer, product):
    pricer.price(product, mkt)
    return 100 * float(product.price) / K0, 100 * float(getattr(product, "standard_error", 0.0)) / K0

## Autocallable template
No coupons (rate 0), autocall at $10^6\%$, put barrier 80%: the price is minus the put (investor's side).

In [ ]:
NEVER = 1e6      # autocall barrier in % of K0: never reached


def autocallable_template(fixing_dates, coupon, coupon_barrier, put_barrier):
    # AutocallableOption per unit notional; autocall disabled, the put active only if put_barrier > 0
    n = len(fixing_dates)
    return tq.AutocallableOption(
        ccy=eq.currency, notional=1.0, start_date=EVALUATION_DATE, end_date=fixing_dates[-1], strike=K0,
        coupon_fixing_dates=fixing_dates, coupon_payment_dates=fixing_dates,
        coupon_rates=[coupon] * n, coupon_barriers=[coupon_barrier] * n, memory=False,
        autocall_fixing_dates=fixing_dates, autocall_payment_dates=fixing_dates, autocall_barrier=[NEVER] * n,
        payoff_barrier=put_barrier, payoff_participation=1.0, payoff_type="put", underlying=TICKER)

In [ ]:
template = autocallable_template([expiry], 0.0, 0.0, put_barrier=BARRIER_PCT)
ac_pricer = tq.AutocallableMCPricer(model=tq.LocalVolatilityModel, n_paths=100_000, barrier_smoothing=0.0)
ac_pricer.price(template, mkt)
ac_put, ac_se = -100 * float(template.price), 100 * float(template.standard_error)
print(f"autocallable template (LV): put {ac_put:.3f}%  +/- {ac_se:.3f}%")

## `BarrierOption` with the barrier at expiry
`monitoring_dates=[expiry]`: Black–Scholes exact as $\mathrm{Put}(H) + (K_0 - H)\,\mathrm{DigitalPut}(H)$ with the single vol at $K_0$,
MC local vol on the same payoff.

In [ ]:
at_expiry = {
    "BS": pct(tq.BlackScholesPricer(), dip([expiry])),
    "MC LV": pct(tq.VanillaMCPricer(model=tq.LocalVolatilityModel, n_paths=100_000, n_steps=260, digital_smoothing=0.0),
                 dip([expiry])),
    "autocallable LV": (ac_put, ac_se),
}
print(pd.DataFrame(at_expiry, index=["% notional", "s.e."]).T.round(3).to_string())

## Monitoring: at expiry vs discrete vs continuous
The same put with a real barrier: knocked in if the spot touches $H$ on any monitoring date (or at any time). Black–Scholes:
Reiner–Rubinstein, discrete dates by the Broadie–Glasserman–Kou shift; MC: dates on the grid, continuous by the Brownian
bridge between weekly steps.

In [ ]:
schedule = tq.ScheduleGenerator(calendar, mf)
monitoring = {
    "at expiry": [expiry],
    "annual": schedule.generate(EVALUATION_DATE, expiry, 12, tq.TimeUnit.Months)[1:],
    "monthly": schedule.generate(EVALUATION_DATE, expiry, 1, tq.TimeUnit.Months)[1:],
    "continuous": None,
}
rows = []
for name, dates in monitoring.items():
    bs, _ = pct(tq.BlackScholesPricer(), dip(dates))
    lv, lv_se = pct(tq.VanillaMCPricer(model=tq.LocalVolatilityModel, n_paths=100_000, n_steps=260,
                                       digital_smoothing=0.0), dip(dates))
    rows.append((name, bs, lv, lv_se))
print(pd.DataFrame(rows, columns=["monitoring", "BS", "MC LV", "s.e."]).set_index("monitoring")
      .round(3).to_string())
vanilla = tq.VanillaOption(eq.currency, start_date=EVALUATION_DATE, end_date=expiry, option_type=tq.OptionType.Put,
                           strike=K0, underlying=TICKER)
print(f"\nvanilla put (BS): {pct(tq.BlackScholesPricer(), vanilla)[0]:.3f}%")

## Reading
- Template = `BarrierOption` at expiry (MC LV) within the Monte Carlo error: the autocallable's put is a European
  knock-in, observed only at $T$.
- More monitoring dates, more chances to touch $H$: at expiry < annual < monthly < continuous $\le$ vanilla. Calling the
  autocallable's put a "down-and-in" with a continuous barrier would overprice it.
- BS vs LV: Black–Scholes uses the single vol at $K_0$; local vol prices the whole smile (the barrier region is
  at higher vols with a negative skew).
- BGK with few dates (annual: 5) is a rough approximation; it improves with the number of dates.